# Rhythmx — Sprint 4: Pretrained Embedding Extraction
### CNN (baseline, from Sprint 3) vs. musicnn vs. MERT

This notebook extracts embeddings from two pretrained audio models — **musicnn**
(music-specific auto-tagging CNN) and **MERT** (self-supervised music foundation model)
— for the same GTZAN clips used to train `GenreCNN`. These embeddings feed lightweight
classifier heads in the companion workflow doc (`Sprint4_Comparison_Workflow.md`),
so we can benchmark our from-scratch CNN against transfer-learning baselines on
identical train/val/test splits.

**Outputs of this notebook:**
- `embeddings/musicnn_{split}.npy` + `embeddings/musicnn_{split}_labels.csv`
- `embeddings/mert_{split}.npy` + `embeddings/mert_{split}_labels.csv`
- `embeddings/extraction_timing.csv` (for the cost-comparison table in Sprint 4)

**Prerequisite:** run this against the exact same file list / split as `CNN_Training.ipynb`
so results are directly comparable. Do not re-split randomly here.


## 1. Environment setup

musicnn is TensorFlow-based; MERT is PyTorch/HuggingFace-based. Rather than fight
dependency conflicts inside your existing `DS_class` conda env (which is tuned for
your PyTorch CNN pipeline), create an isolated env just for embedding extraction.
Run this in your WSL2 Ubuntu terminal:

```bash
conda create -n rhythmx_embed python=3.10 -y
conda activate rhythmx_embed

# musicnn dependency pin (numpy<1.17,>=1.14.5); those numpy versions are incompatible with Python 3.10
# Python 3.10 and other newer packages don't have the distutils/ccompiler.py those older numpy versions depend on
# musicnn (TF-based auto-tagger)
pip install musicnn --no-deps
pip install "numpy>=1.19,<1.24" "tensorflow==3.15.0.post1"

# MERT (HuggingFace)
pip install transformers torch torchaudio accelerate soxr

# shared utilities
pip install "librosa>=0.7.0,<0.9" soundfile audioread pandas numpy scikit-learn psycopg2-binary tqdm "setuptools<81"

# register the env as a Jupyter kernel
pip install ipykernel
python -m ipykernel install --user --name rhythmx_embed --display-name "Python (rhythmx_embed)"
```

Then select the **Python (rhythmx_embed)** kernel for this notebook in VS Code
before running the cells below.

> Note: MERT-v1-330M is ~330M params and will be slow on CPU. If you don't have GPU
> access in WSL2 (check with `nvidia-smi`), switch to `m-a-p/MERT-v1-95M` below —
> it's noted inline as a swap-in.


In [1]:
import os
import time
import json
import numpy as np
import pandas as pd
import librosa
import torch
from tqdm import tqdm

# musicnn
from musicnn.extractor import extractor as musicnn_extractor

# MERT
from transformers import Wav2Vec2FeatureExtractor, AutoModel
print("transformers audio imports OK")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

/home/winni/miniconda3/envs/rhythmx_embed/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/winni/miniconda3/envs/rhythmx_embed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-11 19:31:57.164083: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-11 19:31:57.328776: I external/local_

transformers audio imports OK
Using device: cpu


## 2. Load the exact same train/val/test split used for `GenreCNN`

Point this at whatever your CNN notebook used as the source of truth — either the
`vw_clean_tracks` Postgres view (if the split/fold was persisted there) or a saved
CSV of file paths + labels + split assignment. **Do not regenerate the split here.**

Adjust the query/path below to match your actual Sprint 3 setup.


In [2]:
# Pull from Postgres (music_genre_db) ---
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    dbname="music_genre_db",
    user="postgres",          # adjust to your WSL2 Postgres user
    password=os.environ.get("PGPASSWORD", ""),
    port=5432,
)

query = """
    SELECT track_id, file_path, label AS genre, split
    FROM vw_clean_tracks
    WHERE split IN ('train', 'val', 'test')
    ORDER BY track_id
"""
tracks_df = pd.read_sql(query, conn)
conn.close()

tracks_df.head()

split_counts = tracks_df["split"].value_counts()
print(split_counts)

# Sanity check against Sprint 3's known split sizes
expected_counts = {"train": 677, "val": 141, "test": 153}
for split_name, expected_n in expected_counts.items():
    actual_n = split_counts.get(split_name, 0)
    assert actual_n == expected_n, (
        f"Split mismatch for '{split_name}': expected {expected_n}, got {actual_n}. "
        "vw_clean_tracks may have changed since Sprint 3 — investigate before extracting embeddings."
    )

print("Split counts match Sprint 3 exactly — safe to proceed.")
tracks_df.head()

split
train    677
test     153
val      141
Name: count, dtype: int64
Split counts match Sprint 3 exactly — safe to proceed.


/tmp/ipykernel_39220/1076542290.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  tracks_df = pd.read_sql(query, conn)


,track_id,file_path,genre,split
0,1,../Data_Music/processed/blues/blues.00000.wav,blues,val
1,2,../Data_Music/processed/blues/blues.00001.wav,blues,train
2,3,../Data_Music/processed/blues/blues.00002.wav,blues,train
3,4,../Data_Music/processed/blues/blues.00003.wav,blues,test
4,5,../Data_Music/processed/blues/blues.00004.wav,blues,test


## 3. musicnn embedding extraction

`musicnn.extractor` returns penultimate-layer features per clip (taggram + pooled
embedding). We use the pooled `features['mean_pool']` (or `features['max_pool']`)
representation as our fixed-length embedding — 200-dim for the `MSD_musicnn` model,
753-dim for the `MTT_musicnn` model. `MSD_musicnn` (trained on the Million Song
Dataset) is the closer match to a genre-classification task; start there.


In [3]:
def extract_musicnn_embedding(file_path, model="MSD_musicnn"):
    """Return a single fixed-length embedding vector for one audio clip."""
    taggram, tags, features = musicnn_extractor(
        file_path, model=model, extract_features=True
    )
    # mean_pool: (n_frames, 200) -> average over time for a clip-level vector
    embedding = features["mean_pool"].mean(axis=0)
    return embedding


def extract_musicnn_split(df, split_name, out_dir="embeddings"):
    os.makedirs(out_dir, exist_ok=True)
    split_df = df[df["split"] == split_name].reset_index(drop=True)

    embeddings = []
    labels = []
    failures = []
    start = time.time()

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"musicnn:{split_name}"):
        try:
            emb = extract_musicnn_embedding(row["file_path"])
            embeddings.append(emb)
            labels.append(row["genre"])
        except Exception as e:
            failures.append((row["file_path"], str(e)))

    elapsed = time.time() - start
    embeddings = np.stack(embeddings)

    np.save(f"{out_dir}/musicnn_{split_name}.npy", embeddings)
    pd.DataFrame({"genre": labels}).to_csv(
        f"{out_dir}/musicnn_{split_name}_labels.csv", index=False
    )

    if failures:
        print(f"  {len(failures)} clips failed extraction — see failures list")

    return {
        "model": "musicnn",
        "split": split_name,
        "n_clips": len(embeddings),
        "embedding_dim": embeddings.shape[1],
        "total_seconds": elapsed,
        "seconds_per_clip": elapsed / max(len(embeddings), 1),
        "n_failures": len(failures),
    }


musicnn_timing = []
for split in ["train", "val", "test"]:
    musicnn_timing.append(extract_musicnn_split(tracks_df, split))

pd.DataFrame(musicnn_timing)

musicnn:train:   0%|          | 0/677 [00:00<?, ?it/s]

/home/winni/miniconda3/envs/rhythmx_embed/lib/python3.10/site-packages/musicnn/models.py:58: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.BatchNormalization` instead. In particular, `tf.control_dependencies(tf.GraphKeys.UPDATE_OPS)` should not be used (consult the `tf.keras.layers.BatchNormalization` documentation).
  normalized_input = tf.compat.v1.layers.batch_normalization(expand_input, training=is_training)
/home/winni/miniconda3/envs/rhythmx_embed/lib/python3.10/site-packages/musicnn/models.py:103: UserWarning: `tf.layers.conv2d` is deprecated and will be removed in a future version. Please Use `tf.keras.layers.Conv2D` instead.
  conv = tf.compat.v1.layers.conv2d(inputs=inputs,
/home/winni/miniconda3/envs/rhythmx_embed/lib/python3.10/site-packages/musicnn/models.py:108: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Bat

Computing spectrogram (w/ librosa) and tags (w/ tensorflow).. 

musicnn:train:   0%|          | 1/677 [00:03<40:02,  3.55s/it]

done!


...

musicnn:test:  99%|█████████▉| 152/153 [03:50<00:01,  1.48s/it]

done!
Computing spectrogram (w/ librosa) and tags (w/ tensorflow).. 

musicnn:test: 100%|██████████| 153/153 [03:52<00:00,  1.52s/it]

done!


,model,split,n_clips,embedding_dim,total_seconds,seconds_per_clip,n_failures
0,musicnn,train,677,753,1377.515565,2.034735,0
1,musicnn,val,141,753,218.166939,1.547283,0
2,musicnn,test,153,753,232.381675,1.518834,0


## 4. MERT embedding extraction

MERT expects **24 kHz mono audio**, so clips are resampled from GTZAN's native
22.05 kHz. We mean-pool across the time dimension of the last hidden state to get
a clip-level vector. `MERT-v1-330M` outputs 1024-dim embeddings; the smaller
`MERT-v1-95M` outputs 768-dim — swap the `MODEL_NAME` if you're CPU-bound.


In [ ]:
MODEL_NAME = "m-a-p/MERT-v1-95M"   # swap to "m-a-p/MERT-v1-330M" if have GPU for faster extraction
TARGET_SR = 24000

mert_processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME, trust_remote_code=True)
mert_model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE)
mert_model.eval()


def extract_mert_embedding(file_path):
    """Return a single fixed-length embedding vector for one audio clip."""
    waveform, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)

    inputs = mert_processor(
        waveform, sampling_rate=TARGET_SR, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = mert_model(**inputs, output_hidden_states=True)

    # Mean-pool the final hidden state over the time dimension
    last_hidden = outputs.hidden_states[-1].squeeze(0)   # (time, hidden_dim)
    embedding = last_hidden.mean(dim=0).cpu().numpy()
    return embedding


def extract_mert_split(df, split_name, out_dir="embeddings"):
    os.makedirs(out_dir, exist_ok=True)
    split_df = df[df["split"] == split_name].reset_index(drop=True)

    embeddings = []
    labels = []
    failures = []
    start = time.time()

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"MERT:{split_name}"):
        try:
            emb = extract_mert_embedding(row["file_path"])
            embeddings.append(emb)
            labels.append(row["genre"])
        except Exception as e:
            failures.append((row["file_path"], str(e)))

    elapsed = time.time() - start
    embeddings = np.stack(embeddings)

    np.save(f"{out_dir}/mert_{split_name}.npy", embeddings)
    pd.DataFrame({"genre": labels}).to_csv(
        f"{out_dir}/mert_{split_name}_labels.csv", index=False
    )

    if failures:
        print(f"  {len(failures)} clips failed extraction — see failures list")

    return {
        "model": "mert",
        "split": split_name,
        "n_clips": len(embeddings),
        "embedding_dim": embeddings.shape[1],
        "total_seconds": elapsed,
        "seconds_per_clip": elapsed / max(len(embeddings), 1),
        "n_failures": len(failures),
    }


mert_timing = []
for split in ["train", "val", "test"]:
    mert_timing.append(extract_mert_split(tracks_df, split))

pd.DataFrame(mert_timing)

## 5. Consolidate timing + sanity checks

Save extraction timing for both models — this feeds the cost/latency comparison
table in the Sprint 4 writeup (accuracy isn't the only axis your capstone should
compare on).


In [ ]:
timing_df = pd.DataFrame(musicnn_timing + mert_timing)
os.makedirs("embeddings", exist_ok=True)
timing_df.to_csv("embeddings/extraction_timing.csv", index=False)
timing_df

# Quick sanity check: confirm labels line up across splits/models before moving on
for model in ["musicnn", "mert"]:
    for split in ["train", "val", "test"]:
        emb = np.load(f"embeddings/{model}_{split}.npy")
        lbl = pd.read_csv(f"embeddings/{model}_{split}_labels.csv")
        assert emb.shape[0] == len(lbl), f"Mismatch: {model}/{split}"
        print(f"{model:8s} {split:5s} -> embeddings {emb.shape}, labels {len(lbl)}")

---
**Next step:** open `Sprint4_Comparison_Workflow.md` to train classifier heads on
these embeddings, evaluate against your Sprint 3 CNN test metrics, and build the
final 3-way comparison table.
